# 부산 아파트 실거래가 데이터 전처리

## 목적

데이터 탐색 단계에서 확인한 문제와 처리 기준을 바탕으로
원본 데이터를 분석 가능한 형태로 정제한다.

주요 전처리 항목은 다음과 같다.

- 거래금액 문자열 → 숫자형 변환
- 계약년월 + 계약일 → 계약일자 생성
- 해제사유발생일 → 날짜형 변환
- 등기일자 → 날짜형 변환
- `-` 값의 의미에 따른 처리
- 중복 후보 데이터 유지
- 이상치 후보 데이터 유지
- 전처리 결과 검증
- 정제 데이터 저장

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_PATH = Path("../data/raw/부산_아파트(매매)_실거래가_202508_202607.csv")

df_raw = pd.read_csv(
    DATA_PATH,
    encoding="cp949",
    skiprows=15
)

df_raw.shape

(38163, 20)

## 원본 데이터 보존

전처리 과정에서 원본 DataFrame을 직접 수정하지 않고,
복사본을 생성하여 정제 작업을 수행한다.

이를 통해 전처리 전후 데이터를 비교하고
필요할 경우 원본 상태로 다시 확인할 수 있도록 한다.

In [3]:
df = df_raw.copy()

In [4]:
print("원본:", df_raw.shape)
print("전처리용:", df.shape)

원본: (38163, 20)
전처리용: (38163, 20)


## 1. 거래금액 자료형 변환

`거래금액(만원)`은 금액 데이터이지만
천 단위 구분 쉼표(`,`)가 포함되어 있어 문자열(`str`)로 저장되어 있다.

탐색 단계에서 쉼표를 제거하면 전체 데이터가 숫자형으로
정상 변환 가능한 것을 확인했다.

따라서 쉼표를 제거한 뒤 정수형으로 변환한다.

In [5]:
df["거래금액(만원)"] = (
    df["거래금액(만원)"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

In [6]:
df["거래금액(만원)"].head()

0    39000
1    33000
2     8200
3    35000
4    43800
Name: 거래금액(만원), dtype: int64

In [7]:
df["거래금액(만원)"].dtype

dtype('int64')

In [8]:
pd.DataFrame({
    "원본": df_raw["거래금액(만원)"].head(10),
    "변환 후": df["거래금액(만원)"].head(10)
})

,원본,변환 후
0,"39,000",39000
1,"33,000",33000
2,"8,200",8200
3,"35,000",35000
4,"43,800",43800
5,"34,400",34400
6,"20,900",20900
7,"39,000",39000
8,"84,000",84000
9,"34,000",34000


## 2. 계약일자 생성

원본 데이터에서는 계약 날짜가 `계약년월`과 `계약일`로 분리되어 있다.

탐색 단계에서 두 컬럼을 결합했을 때 전체 데이터가 정상적으로
날짜형(`datetime`)으로 변환 가능한 것을 확인했다.

따라서 두 컬럼을 결합하여 분석에 활용할 `계약일자` 컬럼을 생성한다.

In [9]:
df["계약일자"] = pd.to_datetime(
    df["계약년월"].astype(str)
    + df["계약일"].astype(str).str.zfill(2),
    format="%Y%m%d"
)

In [10]:
df[
    ["계약년월", "계약일", "계약일자"]
].head(10)

,계약년월,계약일,계약일자
0,202607,31,2026-07-31
1,202607,31,2026-07-31
2,202607,31,2026-07-31
3,202607,31,2026-07-31
4,202607,31,2026-07-31
5,202607,31,2026-07-31
6,202607,31,2026-07-31
7,202607,31,2026-07-31
8,202607,31,2026-07-31
9,202607,31,2026-07-31


In [11]:
df["계약일자"].dtype

dtype('<M8[us]')

In [12]:
df["계약일자"].isna().sum()

np.int64(0)

## 3. 해제사유발생일 날짜형 변환

`해제사유발생일`은 거래가 해제된 경우 `YYYYMMDD` 형식의 날짜가 기록되어 있고,
거래가 해제되지 않은 경우 `-`로 표시되어 있다.

탐색 단계에서 `-`는 단순 오류가 아니라
거래가 해제되지 않아 날짜가 존재하지 않는 상태임을 확인했다.

따라서 `-`는 날짜 없음(`NaT`)으로 처리하고,
나머지 값은 날짜형(`datetime`)으로 변환한다.

In [13]:
df["해제사유발생일"] = pd.to_datetime(
    df["해제사유발생일"].replace("-", pd.NA),
    format="%Y%m%d"
)

In [14]:
df["해제사유발생일"].head(20)

0    NaT
1    NaT
2    NaT
3    NaT
4    NaT
5    NaT
6    NaT
7    NaT
8    NaT
9    NaT
10   NaT
11   NaT
12   NaT
13   NaT
14   NaT
15   NaT
16   NaT
17   NaT
18   NaT
19   NaT
Name: 해제사유발생일, dtype: datetime64[us]

In [15]:
df["해제사유발생일"].dtype

dtype('<M8[us]')

In [16]:
df["해제사유발생일"].isna().sum()

np.int64(35940)

## 4. 등기일자 날짜형 변환

`등기일자`는 `YY.MM.DD` 형식의 문자열로 저장되어 있으며,
등기 정보가 없는 경우 `-`로 표시되어 있다.

탐색 단계에서 날짜 형식을 자동 추론할 경우
잘못된 날짜로 변환되는 문제가 발생했기 때문에,
원본 형식에 맞게 `format="%y.%m.%d"`를 명시하여 변환한다.

`-` 값은 날짜 없음(`NaT`)으로 처리한다.

In [17]:
df["등기일자"] = pd.to_datetime(
    df["등기일자"].replace("-", pd.NA),
    format="%y.%m.%d"
)

In [18]:
df["등기일자"].head(20)

0           NaT
1           NaT
2           NaT
3           NaT
4           NaT
5           NaT
6    2026-08-11
7    2026-07-31
8           NaT
9           NaT
10          NaT
11          NaT
12          NaT
13   2026-08-04
14          NaT
15          NaT
16          NaT
17          NaT
18          NaT
19          NaT
Name: 등기일자, dtype: datetime64[us]

In [19]:
df["등기일자"].dtype

dtype('<M8[us]')

In [20]:
df["등기일자"].isna().sum()

np.int64(7041)

In [21]:
df["등기일자"].min(), df["등기일자"].max()

(Timestamp('2025-08-01 00:00:00'), Timestamp('2026-08-19 00:00:00'))

## 5. 중개사소재지 미기재 값 처리

`중개사소재지`의 `-` 값은 거래유형에 따라 의미가 다름을
탐색 단계에서 확인했다.

- 직거래 + `-`: 중개사가 존재하지 않으므로 정상적인 '해당없음'
- 중개거래 + `-`: 중개거래임에도 정보가 없어 실제 결측값으로 판단

따라서 모든 `-` 값을 동일하게 처리하지 않고
거래유형에 따라 의미를 구분하여 변환한다.

In [22]:
direct_mask = (
    (df["거래유형"] == "직거래") &
    (df["중개사소재지"] == "-")
)

broker_missing_mask = (
    (df["거래유형"] == "중개거래") &
    (df["중개사소재지"] == "-")
)

df.loc[direct_mask, "중개사소재지"] = "해당없음"
df.loc[broker_missing_mask, "중개사소재지"] = pd.NA

In [23]:
df.loc[
    df["거래유형"] == "직거래",
    "중개사소재지"
].value_counts(dropna=False).head()

중개사소재지
해당없음    2767
Name: count, dtype: int64

In [24]:
df["중개사소재지"].isna().sum()

np.int64(1)

In [25]:
(df["중개사소재지"] == "-").sum()

np.int64(0)

## 6. 동 정보 미기재 값 처리

`동` 컬럼에는 9,985건의 `-` 값이 존재했다.

탐색 단계에서 등기일자가 없는 거래에서는 모두 동 정보도 미기재되어 있었으며,
등기일자가 존재하는 거래 중에서도 동 정보가 제공되지 않는 단지가 존재함을 확인했다.

따라서 `-` 값을 데이터 오류로 판단하여 거래 행을 삭제하지 않는다.

다만 `-`는 실제 동 이름이 아니므로,
분석 시 결측값으로 인식할 수 있도록 `pd.NA`로 변환한다.

In [26]:
df["동"] = df["동"].replace("-", pd.NA)

In [27]:
df["동"].isna().sum()

np.int64(9985)

In [28]:
(df["동"] == "-").sum()

np.int64(0)

In [29]:
df.shape

(38163, 21)

## 7. 중간 전처리 결과 검증

지금까지 거래금액, 날짜 데이터, 중개사소재지, 동 정보에 대한
전처리를 적용했다.

다음 단계로 넘어가기 전에 데이터 타입과 결측값 표현을 다시 확인하여
의도한 형태로 정상 변환되었는지 검증한다.

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 38163 entries, 0 to 38162
Data columns (total 21 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   NO        38163 non-null  int64         
 1   시군구       38163 non-null  str           
 2   번지        38163 non-null  str           
 3   본번        38163 non-null  int64         
 4   부번        38163 non-null  int64         
 5   단지명       38163 non-null  str           
 6   전용면적(㎡)   38163 non-null  float64       
 7   계약년월      38163 non-null  int64         
 8   계약일       38163 non-null  int64         
 9   거래금액(만원)  38163 non-null  int64         
 10  동         28178 non-null  str           
 11  층         38163 non-null  int64         
 12  매수자       38163 non-null  str           
 13  매도자       38163 non-null  str           
 14  건축년도      38163 non-null  int64         
 15  도로명       38163 non-null  str           
 16  해제사유발생일   2223 non-null   datetime64[us]
 17  거래유형      38163 non-nul

In [31]:
df.isna().sum()

NO              0
시군구             0
번지              0
본번              0
부번              0
단지명             0
전용면적(㎡)         0
계약년월            0
계약일             0
거래금액(만원)        0
동            9985
층               0
매수자             0
매도자             0
건축년도            0
도로명             0
해제사유발생일     35940
거래유형            0
중개사소재지          1
등기일자         7041
계약일자            0
dtype: int64

In [32]:
string_columns = df.select_dtypes(include=["object", "string"]).columns

hyphen_counts = {
    column: (df[column] == "-").sum()
    for column in string_columns
}

hyphen_counts

{'시군구': np.int64(0),
 '번지': np.int64(0),
 '단지명': np.int64(0),
 '동': np.int64(0),
 '매수자': np.int64(0),
 '매도자': np.int64(0),
 '도로명': np.int64(0),
 '거래유형': np.int64(0),
 '중개사소재지': np.int64(0)}

In [33]:
df.shape

(38163, 21)

### 중간 전처리 검증 결과

지금까지 적용한 전처리 결과를 다시 검증했다.

- 거래금액은 정수형으로 정상 변환되었다.
- 계약일자, 해제사유발생일, 등기일자는 날짜형으로 정상 변환되었다.
- 원본의 `-`로 표현되던 값은 의미에 따라 결측값 또는 `해당없음`으로 정리되었다.
- 문자열 컬럼에서 `-` 값은 더 이상 남아 있지 않았다.
- 전체 행 수는 38,163건으로 원본과 동일하게 유지되었다.

현재까지 데이터 삭제 없이 원본의 의미를 보존하면서
분석 가능한 형태로 자료형과 결측값 표현을 정리했다.

## 8. 분석용 파생변수 생성

정제된 원본 컬럼을 이용하여
이후 분석에 활용할 파생변수를 생성한다.

먼저 `시군구` 컬럼에서 부산의 구 정보를 분리하여
지역별 거래가격과 거래량 분석에 사용할 `구` 컬럼을 생성한다.

In [34]:
df["구"] = df["시군구"].str.split().str[1]

In [35]:
df[["시군구", "구"]].head(10)

,시군구,구
0,부산광역시 부산진구 개금동,부산진구
1,부산광역시 서구 서대신동2가,서구
2,부산광역시 중구 중앙동4가,중구
3,부산광역시 부산진구 가야동,부산진구
4,부산광역시 부산진구 개금동,부산진구
5,부산광역시 부산진구 개금동,부산진구
6,부산광역시 부산진구 부전동,부산진구
7,부산광역시 영도구 동삼동,영도구
8,부산광역시 부산진구 양정동,부산진구
9,부산광역시 동래구 안락동,동래구


In [36]:
df["구"].value_counts()

구
해운대구    5033
부산진구    4848
동래구     3804
북구      3328
연제구     3276
남구      3254
사하구     2820
수영구     2224
금정구     1932
기장군     1753
사상구     1695
강서구     1553
서구       952
영도구      851
동구       710
중구       130
Name: count, dtype: int64

In [37]:
df["구"].isna().sum()

np.int64(0)

### 거래해제여부 파생변수 생성

`해제사유발생일`에 실제 날짜가 존재하면 거래가 해제된 것으로 볼 수 있다.

향후 거래 유지 건과 해제 건을 쉽게 구분할 수 있도록
`해제사유발생일`을 이용하여 `거래해제여부` 파생변수를 생성한다.

- `True`: 거래 해제
- `False`: 거래 유지

In [38]:
df["거래해제여부"] = df["해제사유발생일"].notna()

In [39]:
df[
    ["해제사유발생일", "거래해제여부"]
].head(20)

,해제사유발생일,거래해제여부
0,NaT,False
1,NaT,False
2,NaT,False
3,NaT,False
4,NaT,False
5,NaT,False
6,NaT,False
7,NaT,False
8,NaT,False
9,NaT,False


In [40]:
df.loc[
    df["거래해제여부"],
    ["계약일자", "해제사유발생일", "거래해제여부"]
].head(10)

,계약일자,해제사유발생일,거래해제여부
77,2026-07-31,2026-08-20,True
89,2026-07-30,2026-08-06,True
217,2026-07-29,2026-08-05,True
286,2026-07-28,2026-08-10,True
304,2026-07-28,2026-07-31,True
376,2026-07-27,2026-07-30,True
432,2026-07-25,2026-07-30,True
605,2026-07-23,2026-07-24,True
618,2026-07-23,2026-08-04,True
661,2026-07-22,2026-07-27,True


In [41]:
df["거래해제여부"].value_counts()

거래해제여부
False    35940
True      2223
Name: count, dtype: int64

In [42]:
df["거래해제여부"].sum()

np.int64(2223)

### 전용면적(평) 및 평당가격 파생변수 생성

원본 데이터의 면적은 제곱미터(㎡) 단위로 제공된다.

부동산 가격을 면적과 함께 비교하기 쉽도록
전용면적을 평 단위로 변환한 `전용면적(평)` 컬럼을 생성한다.

또한 거래금액을 전용면적(평)으로 나누어
`평당가격(만원)` 컬럼을 생성한다.

※ 본 프로젝트의 평당가격은 공급면적이 아닌
`전용면적`을 기준으로 계산한다.

In [43]:
df["전용면적(평)"] = df["전용면적(㎡)"] / 3.305785

df["평당가격(만원)"] = (
    df["거래금액(만원)"] / df["전용면적(평)"]
)

In [44]:
df[
    [
        "전용면적(㎡)",
        "전용면적(평)",
        "거래금액(만원)",
        "평당가격(만원)"
    ]
].head(10)

,전용면적(㎡),전용면적(평),거래금액(만원),평당가격(만원)
0,118.4700,35.837176,39000,1088.255381
1,84.9750,25.704938,33000,1283.800000
2,68.9200,20.848301,8200,393.317426
3,71.6577,21.676455,35000,1614.655159
4,134.9200,40.813302,43800,1073.179536
5,84.9900,25.709476,34400,1338.028050
6,71.0280,21.485971,20900,972.727748
7,84.9133,25.686274,39000,1518.320628
8,84.9268,25.690358,84000,3269.709208
9,84.7700,25.642926,34000,1325.901734


In [45]:
df[
    [
        "전용면적(㎡)",
        "전용면적(평)",
        "거래금액(만원)",
        "평당가격(만원)"
    ]
].head(10).round(2)

,전용면적(㎡),전용면적(평),거래금액(만원),평당가격(만원)
0,118.47,35.84,39000,1088.26
1,84.98,25.70,33000,1283.80
2,68.92,20.85,8200,393.32
3,71.66,21.68,35000,1614.66
4,134.92,40.81,43800,1073.18
5,84.99,25.71,34400,1338.03
6,71.03,21.49,20900,972.73
7,84.91,25.69,39000,1518.32
8,84.93,25.69,84000,3269.71
9,84.77,25.64,34000,1325.90


In [46]:
df[
    ["전용면적(평)", "평당가격(만원)"]
].describe()

,전용면적(평),평당가격(만원)
count,38163.000000,38163.000000
mean,23.346294,1862.877576
std,7.328848,1047.276989
min,3.861231,136.227404
25%,18.127313,1087.724064
50%,25.031876,1656.082609
75%,25.703426,2351.382273
max,74.066825,9176.475358


In [47]:
df[
    ["전용면적(평)", "평당가격(만원)"]
].isna().sum()

전용면적(평)     0
평당가격(만원)    0
dtype: int64

### 건물연식 파생변수 생성

건축년도만으로는 거래 당시 건물이 얼마나 오래되었는지
직관적으로 비교하기 어렵다.

따라서 `계약일자`의 연도와 `건축년도`를 이용하여
거래 당시의 건물 연식을 나타내는 `건물연식` 컬럼을 생성한다.

건물연식 = 계약연도 - 건축년도

In [48]:
df["건물연식"] = (
    df["계약일자"].dt.year - df["건축년도"]
)

In [49]:
df[
    ["계약일자", "건축년도", "건물연식"]
].head(10)

,계약일자,건축년도,건물연식
0,2026-07-31,1998,28
1,2026-07-31,2005,21
2,2026-07-31,1976,50
3,2026-07-31,2005,21
4,2026-07-31,1998,28
5,2026-07-31,1998,28
6,2026-07-31,1999,27
7,2026-07-31,2022,4
8,2026-07-31,2025,1
9,2026-07-31,2000,26


In [50]:
df["건물연식"].describe()

count    38163.000000
mean        17.857847
std         11.795321
min          0.000000
25%          7.000000
50%         17.000000
75%         28.000000
max         64.000000
Name: 건물연식, dtype: float64

In [51]:
(df["건물연식"] < 0).sum()

np.int64(0)

In [52]:
df["건물연식"].isna().sum()

np.int64(0)

## 9. 최종 컬럼 구성 정리

전처리 및 파생변수 생성을 완료한 후
분석에 사용하기 쉽도록 컬럼의 순서를 정리한다.

원본 데이터의 주요 정보는 유지하고,
추가로 생성한 계약일자, 구, 거래해제여부,
전용면적(평), 평당가격, 건물연식 등의 파생변수를 함께 포함한다.

`NO` 컬럼은 원본 데이터의 행을 추적할 수 있도록 유지한다.

In [53]:
final_columns = [
    "NO",
    "계약일자",
    "계약년월",
    "계약일",
    "구",
    "시군구",
    "번지",
    "본번",
    "부번",
    "단지명",
    "전용면적(㎡)",
    "전용면적(평)",
    "층",
    "거래금액(만원)",
    "평당가격(만원)",
    "건축년도",
    "건물연식",
    "동",
    "매수자",
    "매도자",
    "도로명",
    "거래유형",
    "중개사소재지",
    "거래해제여부",
    "해제사유발생일",
    "등기일자"
]

df_processed = df[final_columns].copy()

In [54]:
df_processed.head()

,NO,계약일자,계약년월,계약일,구,시군구,번지,본번,부번,단지명,...,건물연식,동,매수자,매도자,도로명,거래유형,중개사소재지,거래해제여부,해제사유발생일,등기일자
0,1,2026-07-31,202607,31,부산진구,부산광역시 부산진구 개금동,455,455,0,엘지신개금(2-1),...,28,NaN,개인,개인,백양대로300번길 20,중개거래,부산 부산진구,False,NaT,NaT
1,2,2026-07-31,202607,31,서구,부산광역시 서구 서대신동2가,270,270,0,보람,...,21,NaN,개인,개인,대영로 24,직거래,해당없음,False,NaT,NaT
2,3,2026-07-31,202607,31,중구,부산광역시 중구 중앙동4가,40-15,40,15,동광맨션,...,50,NaN,개인,법인,동광길 66,중개거래,부산 동구,False,NaT,NaT
3,4,2026-07-31,202607,31,부산진구,부산광역시 부산진구 가야동,588,588,0,반도보라빌,...,21,NaN,개인,개인,가야공원로 41,중개거래,부산 부산진구,False,NaT,NaT
4,5,2026-07-31,202607,31,부산진구,부산광역시 부산진구 개금동,455,455,0,엘지신개금(2-1),...,28,NaN,개인,개인,백양대로300번길 20,중개거래,부산 부산진구,False,NaT,NaT


In [55]:
df_processed.shape

(38163, 26)

## 10. 최종 전처리 결과 검증

정제 데이터 저장 전에 전체 행 수, 컬럼 수,
자료형 및 주요 결측값을 다시 확인한다.

전처리 과정에서 의도하지 않은 데이터 삭제나
자료형 변환 오류가 발생하지 않았는지 최종 검증한다.

In [56]:
print("원본 데이터:", df_raw.shape)
print("정제 데이터:", df_processed.shape)

원본 데이터: (38163, 20)
정제 데이터: (38163, 26)


In [57]:
df_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 38163 entries, 0 to 38162
Data columns (total 26 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   NO        38163 non-null  int64         
 1   계약일자      38163 non-null  datetime64[us]
 2   계약년월      38163 non-null  int64         
 3   계약일       38163 non-null  int64         
 4   구         38163 non-null  object        
 5   시군구       38163 non-null  str           
 6   번지        38163 non-null  str           
 7   본번        38163 non-null  int64         
 8   부번        38163 non-null  int64         
 9   단지명       38163 non-null  str           
 10  전용면적(㎡)   38163 non-null  float64       
 11  전용면적(평)   38163 non-null  float64       
 12  층         38163 non-null  int64         
 13  거래금액(만원)  38163 non-null  int64         
 14  평당가격(만원)  38163 non-null  float64       
 15  건축년도      38163 non-null  int64         
 16  건물연식      38163 non-null  int64         
 17  동         28178 non-nul

## 11. 전처리 데이터 저장 및 재검증

전처리가 완료된 데이터를 `data/processed` 폴더에 별도 CSV 파일로 저장한다.

원본 데이터는 수정하지 않고 유지하며,
정제 데이터는 UTF-8 형식으로 저장하여 이후 분석 및 시각화 단계에서 활용한다.

저장 후 파일을 다시 불러와 행·열 개수와 주요 데이터가 정상적으로 유지되는지 검증한다.

In [58]:
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = PROCESSED_DIR / "busan_apartment_trades_cleaned_202508_202607.csv"

In [59]:
df_processed.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

In [60]:
OUTPUT_PATH.exists()

True

## 12. 저장 데이터 재검증

전처리 완료 데이터를 CSV로 저장한 뒤 다시 불러와
행·열 개수와 주요 컬럼의 값이 정상적으로 유지되었는지 확인한다.

CSV 파일은 자료형 정보를 직접 저장하지 않으므로,
날짜 컬럼은 다시 불러올 때 날짜형으로 지정한다.

In [61]:
df_check = pd.read_csv(
    OUTPUT_PATH,
    encoding="utf-8-sig",
    parse_dates=[
        "계약일자",
        "해제사유발생일",
        "등기일자"
    ]
)

In [62]:
print("저장 전:", df_processed.shape)
print("저장 후:", df_check.shape)

저장 전: (38163, 26)
저장 후: (38163, 26)


In [63]:
print(
    "거래금액 합계 일치:",
    df_processed["거래금액(만원)"].sum()
    == df_check["거래금액(만원)"].sum()
)

print(
    "거래해제 건수 일치:",
    df_processed["거래해제여부"].sum()
    == df_check["거래해제여부"].sum()
)

print(
    "동 결측치 개수 일치:",
    df_processed["동"].isna().sum()
    == df_check["동"].isna().sum()
)

거래금액 합계 일치: True
거래해제 건수 일치: True
동 결측치 개수 일치: True


In [64]:
df_check.info()

<class 'pandas.DataFrame'>
RangeIndex: 38163 entries, 0 to 38162
Data columns (total 26 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   NO        38163 non-null  int64         
 1   계약일자      38163 non-null  datetime64[us]
 2   계약년월      38163 non-null  int64         
 3   계약일       38163 non-null  int64         
 4   구         38163 non-null  str           
 5   시군구       38163 non-null  str           
 6   번지        38163 non-null  str           
 7   본번        38163 non-null  int64         
 8   부번        38163 non-null  int64         
 9   단지명       38163 non-null  str           
 10  전용면적(㎡)   38163 non-null  float64       
 11  전용면적(평)   38163 non-null  float64       
 12  층         38163 non-null  int64         
 13  거래금액(만원)  38163 non-null  int64         
 14  평당가격(만원)  38163 non-null  float64       
 15  건축년도      38163 non-null  int64         
 16  건물연식      38163 non-null  int64         
 17  동         28178 non-nul

## 13. 전처리 완료 및 재현성 확인

원본 데이터 탐색에서 수립한 처리 기준을 바탕으로
38,163건의 아파트 실거래 데이터를 정제했다.

주요 처리 내용은 다음과 같다.

- 거래금액 문자열을 숫자형으로 변환
- 계약년월과 계약일을 결합하여 계약일자 생성
- 해제사유발생일과 등기일자를 날짜형으로 변환
- `-`로 표현된 값을 데이터의 의미에 따라 결측값 또는 `해당없음`으로 처리
- 거래해제여부, 구, 전용면적(평), 평당가격, 건물연식 파생변수 생성
- 중복 후보와 이상치 후보는 실제 거래 가능성을 고려하여 삭제하지 않음
- 원본 38,163건을 유지한 상태로 26개 컬럼의 정제 데이터셋 생성

정제 데이터는 `data/processed`에 별도로 저장하여
원본 데이터와 전처리 결과를 분리하였다.